# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

## Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed in the environment
!pip install mlcroissant

## 1. Data Loading

We begin by importing required libraries and loading the metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nPublished: {metadata.datePublished}\nVersion: {metadata.version}")

## 2. Data Overview

Explore available record sets and their fields. For datasets with multiple tables or record sets, each is referenced by its `@id`.

> Note: All dataset entities (record sets, fields/columns) are referenced by their Croissant `@id` field for reproducibility and traceability.

Let's list available record sets, their IDs, and their columns. 

In [ ]:
print("Available Record Sets: (by @id)\n-------------------------")
record_sets = []
# Dataset.record_sets yields RecordSet objects
for rs in dataset.record_sets():
    print(f"- @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    fields = list(rs.fields)
    print(f"  Number of fields: {len(fields)}")
    print("  Fields (by @id):")
    for field in fields:
        print(f"    - {field.id} (name: {field.name}, dataType: {field.data_type})")
    print("")
    record_sets.append(rs.id)

if not record_sets:
    print("No record sets found! Please check the dataset schema or consult documentation.")

## 3. Data Extraction

Let's extract data from each record set. We'll load the tabular data in each record set (referenced by `@id`) into a pandas DataFrame for further analysis.

Replace `<choose_record_set_id>` with the appropriate `@id` from the overview above for focused exploration.

In [ ]:
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded records from record set: {rs_id} (rows: {len(dataframes[rs_id])})")
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
    else:
        print(f"No records found in RecordSet {rs_id}. Skipping...")

if dataframes:
    # Choose the first available table for demonstration
    main_record_set = list(dataframes.keys())[0]
    print(f"\nShowing first few rows of DataFrame for record set: {main_record_set}")
    display(dataframes[main_record_set].head())
else:
    print("No record sets have data!")

## 4. Exploratory Data Analysis (EDA)

Now we'll perform EDA on the selected record set (by `@id`). Common EDA steps include filtering records, normalizing values, and grouping data. 

We'll pick a numeric field (referenced by `@id` and column name) for demonstration. Please adjust field names if needed based on your dataset.

In [ ]:
# Choose the main DataFrame (based on previous extraction)
df = dataframes[main_record_set]
# Print available columns
print(f"Available columns: {df.columns.tolist()}")

# Select a numeric field by examining its name (e.g., 'Molecular Weight', 'Abundance', etc.)
# Here we use the first numeric column found as an example.
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    print("No numeric fields detected in this record set.")
else:
    print(f"Using numeric field for EDA: {numeric_field}")

    # Filtering: keep rows with value > threshold
    threshold = df[numeric_field].mean()  # e.g., mean as threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered rows where {numeric_field} > {threshold:.2f} (kept {len(filtered_df)} rows)")
    print(filtered_df.head())

    # Normalize the field (Z-score)
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} (Z-score):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a string/categorical column if available
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        print(f"\nGrouping results by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df.head())
    else:
        print("No categorical string field available for grouping.")

## 5. Visualization

Let's visualize the distribution of the numeric field and any relevant relationships in the data using plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a grouping field exists, visualize grouped means
    if group_field:
        plt.figure(figsize=(10, 4))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:15]
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Mean {numeric_field} by {group_field} (Top 15)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load a dataset defined by a Croissant schema using the `mlcroissant` library
- Discover and reference record sets and fields by their `@id`
- Extract and manipulate tabular data using pandas
- Perform filtering, normalization, grouping, and visualization

This workflow can be adapted to other Croissant-compatible datasets for reproducible and transparent data analysis leveraging semantic dataset metadata. For deeper insight, consult the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) and the data provider's metadata for domain-specific context.
